# 03 – Band Power & Alpha Detection

Compute spectral band powers using **Welch's method** and track alpha (8–12 Hz) over time.

| Band | Range | Typical association |
|------|-------|--------------------|
| Delta (δ) | 1–4 Hz | Deep sleep |
| Theta (θ) | 4–8 Hz | Drowsiness, meditation |
| Alpha (α) | 8–12 Hz | Relaxed, eyes closed |
| Beta (β) | 12–30 Hz | Active thinking |
| Gamma (γ) | 30–100 Hz | High-level cognition |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

SAMPLE_RATE = 250

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (12, 30),
    "gamma": (30, 100),
}

## Load data

In [ ]:
import glob

csv_files = sorted(glob.glob("../recordings/pieeg_*.csv"))
df = pd.read_csv(csv_files[-1])
df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
ch_cols = [c for c in df.columns if c.startswith("ch")]
print(f"{len(ch_cols)} channels, {len(df)} samples, {df['time'].iloc[-1]:.1f} s")

## (Alternative) Stream live from PiEEG

In [ ]:
# import asyncio, json, websockets
#
# async def collect(seconds=30):
#     rows = []
#     async with websockets.connect("ws://localhost:1616") as ws:
#         welcome = json.loads(await ws.recv())
#         target = seconds * SAMPLE_RATE
#         while len(rows) < target:
#             msg = json.loads(await ws.recv())
#             if "channels" in msg:
#                 rows.append([msg["t"]] + msg["channels"])
#     cols = ["timestamp"] + [f"ch{i+1}" for i in range(welcome["channels"])]
#     return pd.DataFrame(rows, columns=cols)
#
# df = await collect(30)
# df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
# ch_cols = [c for c in df.columns if c.startswith("ch")]

## Power spectral density (PSD) – single channel

In [ ]:
CH = "ch1"  # change to any channel

freqs, psd = signal.welch(df[CH].values, fs=SAMPLE_RATE, nperseg=SAMPLE_RATE * 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd, color="steelblue", lw=1)

colors = {"delta": "#9b59b6", "theta": "#3498db", "alpha": "#2ecc71",
          "beta": "#e67e22", "gamma": "#e74c3c"}
for band, (lo, hi) in BANDS.items():
    mask = (freqs >= lo) & (freqs <= hi)
    ax.fill_between(freqs[mask], psd[mask], alpha=0.3, label=band, color=colors[band])

ax.set_xlim(0, 50)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (µV²/Hz)")
ax.set_title(f"Power Spectral Density – {CH}")
ax.legend()
plt.tight_layout()
plt.show()

## Band power helper

In [ ]:
def band_power(data, fs, band, nperseg=None):
    """Absolute band power (µV²) via Welch."""
    if nperseg is None:
        nperseg = min(len(data), fs * 2)
    freqs, psd = signal.welch(data, fs=fs, nperseg=nperseg)
    lo, hi = band
    mask = (freqs >= lo) & (freqs <= hi)
    return np.trapz(psd[mask], freqs[mask])


def all_band_powers(data, fs):
    """Return dict of {band_name: power}."""
    return {name: band_power(data, fs, rng) for name, rng in BANDS.items()}

## Band power bar chart (all channels)

In [ ]:
powers = {ch: all_band_powers(df[ch].values, SAMPLE_RATE) for ch in ch_cols}
bp_df = pd.DataFrame(powers).T

bp_df.plot.bar(stacked=True, figsize=(12, 5),
               color=[colors[b] for b in BANDS])
plt.ylabel("Power (µV²)")
plt.title("Band Power by Channel")
plt.legend(title="Band")
plt.tight_layout()
plt.show()

## Alpha power over time (sliding window)

Slide a 2-second window and compute alpha power to see when a subject relaxes or closes their eyes.

In [ ]:
WIN_SEC = 2
STEP_SEC = 0.5
win_samples = int(WIN_SEC * SAMPLE_RATE)
step_samples = int(STEP_SEC * SAMPLE_RATE)

data = df[CH].values
times, alpha_vals, beta_vals = [], [], []

start = 0
while start + win_samples <= len(data):
    segment = data[start : start + win_samples]
    times.append(df["time"].values[start] + WIN_SEC / 2)
    alpha_vals.append(band_power(segment, SAMPLE_RATE, BANDS["alpha"]))
    beta_vals.append(band_power(segment, SAMPLE_RATE, BANDS["beta"]))
    start += step_samples

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times, alpha_vals, label="Alpha (8–12 Hz)", color=colors["alpha"], lw=1.5)
ax.plot(times, beta_vals, label="Beta (12–30 Hz)", color=colors["beta"], lw=1.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Power (µV²)")
ax.set_title(f"Alpha vs Beta over time – {CH}")
ax.legend()
plt.tight_layout()
plt.show()

## Alpha / Beta ratio

A rising α/β ratio typically indicates a more relaxed state.

In [ ]:
ratio = np.array(alpha_vals) / (np.array(beta_vals) + 1e-9)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(times, ratio, color="#2c3e50", lw=1.5)
ax.axhline(np.median(ratio), color="red", ls="--", lw=0.8, label="Median")
ax.set_xlabel("Time (s)")
ax.set_ylabel("α / β")
ax.set_title("Alpha / Beta Ratio")
ax.legend()
plt.tight_layout()
plt.show()